In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"  # 請替換為您的GPU編號

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    DataCollatorWithPadding
)
import shap
import os

# 設定隨機種子以確保結果可重現
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# ==========================================
# 1. 設定與準備
# ==========================================
# 模型名稱 (可在此更換為其他 BERT 模型)
MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"  # 請替換為您的模型名稱
DATA_FILE = "final_dataset.csv"  # 請替換為您的檔案路徑

# 檢查是否有 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 定義 Dataset 類別
class ClinicalDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        item = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            max_length=self.max_len,
            padding="max_length"  # 動態 padding 可在 collator 處理，這裡簡化
        )
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# 定義指標計算函數
def calculate_comprehensive_metrics(y_true, y_pred, y_prob):
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    # Basic Metrics
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0) # PPV
    recall = recall_score(y_true, y_pred, zero_division=0)       # Sensitivity
    
    # F1 Scores
    pos_f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    neg_f1 = f1_score(y_true, y_pred, pos_label=0, zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    # Advanced Metrics
    # Specificity (SP) = TN / (TN + FP)
    sp = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    # Negative Predictive Value (NPV) = TN / (TN + FN)
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    
    # Positive Predictive Value (PPV) is same as Precision
    ppv = precision
    
    # AUROC & AUPRC
    try:
        auroc = roc_auc_score(y_true, y_prob)
        auprc = average_precision_score(y_true, y_prob)
    except ValueError:
        auroc = 0
        auprc = 0
        
    return {
        "ACC": acc,
        "Precision": precision,
        "Recall": recall,
        "Macro F1": macro_f1,
        "Positive F1": pos_f1,
        "Negative F1": neg_f1,
        "NPV": npv,
        "PPV": ppv,
        "Specificity": sp,
        "AUROC": auroc,
        "AUPRC": auprc,
        "CM": cm
    }

# ==========================================
# 2. 資料載入與切分
# ==========================================
print("Loading data...")
df = pd.read_csv(DATA_FILE) 
# ----------------------------------------------------------------


X = df['text'].values
y = df['target_long_stay'].values

# 切出測試集 (Hold-out Test Set, 20%)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

print(f"Total samples: {len(df)}")
print(f"Train/Val set size: {len(X_train_full)}")
print(f"Test set size: {len(X_test)}")

# ==========================================
# 3. 10-Fold Cross-Validation
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)

fold_results = []

print("\nStarting 10-Fold Cross-Validation...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train_full)):
    print(f"\nProcessing Fold {fold + 1}/10")
    
    # 準備該 Fold 的數據
    X_train_fold, X_val_fold = X_train_full[train_idx], X_train_full[val_idx]
    y_train_fold, y_val_fold = y_train_full[train_idx], y_train_full[val_idx]
    
    train_dataset = ClinicalDataset(X_train_fold, y_train_fold, tokenizer)
    val_dataset = ClinicalDataset(X_val_fold, y_val_fold, tokenizer)
    
    # 初始化模型
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    ).to(device)
    
    # 訓練參數 (請依據您的顯卡記憶體調整 batch_size)
    training_args = TrainingArguments(
        output_dir=f'./results/fold_{fold}',
        num_train_epochs=3,              # 建議設為 3-5
        per_device_train_batch_size=8,   # 若 OOM 請調小
        per_device_eval_batch_size=16,
        eval_strategy="epoch",
        save_strategy="no",              # 節省空間，不存每個 checkpoint
        logging_dir=f'./logs/fold_{fold}',
        logging_steps=50,
        learning_rate=2e-5,
        weight_decay=0.01,
        report_to="none"                 # 不上傳到 wandb
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=DataCollatorWithPadding(tokenizer)
    )
    
    # 訓練
    trainer.train()
    
    # 預測
    predictions = trainer.predict(val_dataset)
    logits = predictions.predictions
    pred_labels = np.argmax(logits, axis=1)
    # Softmax 取得機率 (取 class 1 的機率)
    pred_probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    
    # 計算指標
    metrics = calculate_comprehensive_metrics(y_val_fold, pred_labels, pred_probs)
    fold_results.append(metrics)
    
    # 釋放記憶體
    del model, trainer
    torch.cuda.empty_cache()

# ==========================================
# 4. 輸出統計分析 (Statistical Analysis)
# ==========================================
print("\n" + "="*50)
print("10-Fold Cross-Validation Results")
print("="*50)

metrics_df = pd.DataFrame(fold_results)
# 移除 Confusion Matrix 以便計算平均值
stats_df = metrics_df.drop(columns=['CM'])

# 計算 Mean 與 Std
summary = stats_df.describe().loc[['mean', 'std']]
print(summary.T)

# 顯示 Confusion Matrix (加總所有 Folds 或顯示最後一個)
total_cm = np.sum(metrics_df['CM'].values)
print("\nAggregated Confusion Matrix (Sum over 10 folds):")
print(total_cm)

# ==========================================
# 5. 最終測試與 SHAP 解釋性分析
# ==========================================
print("\n" + "="*50)
print("Final Evaluation on Test Set & SHAP Analysis")
print("="*50)

# 使用完整的 Train 資料重新訓練一個最終模型 (或者您可以選擇效果最好的 Fold 模型)
final_train_dataset = ClinicalDataset(X_train_full, y_train_full, tokenizer)
final_test_dataset = ClinicalDataset(X_test, y_test, tokenizer)

final_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
).to(device)

final_trainer = Trainer(
    model=final_model,
    args=TrainingArguments(output_dir='./results/final', num_train_epochs=3, per_device_train_batch_size=8),
    train_dataset=final_train_dataset
)
final_trainer.train()

# 測試集評估
test_preds = final_trainer.predict(final_test_dataset)
test_logits = test_preds.predictions
test_labels_pred = np.argmax(test_logits, axis=1)
test_probs = torch.nn.functional.softmax(torch.tensor(test_logits), dim=-1).numpy()[:, 1]

final_metrics = calculate_comprehensive_metrics(y_test, test_labels_pred, test_probs)
print("Final Test Set Metrics:")
for k, v in final_metrics.items():
    if k != 'CM':
        print(f"{k}: {v:.4f}")
print(f"Confusion Matrix:\n{final_metrics['CM']}")

# SHAP 分析
# 注意：SHAP 對於 Transformer 非常耗時，這裡只取測試集的一小部分進行演示
print("\nGenerating SHAP values...")

# 建立一個 Pipeline 供 SHAP 使用
from transformers import pipeline
pred_pipeline = pipeline(
    "text-classification",
    model=final_model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
    return_all_scores=True,
    truncation=True,  # 強制截斷
    max_length=512    # 限制最大長度與模型一致
)

# 定義 SHAP Explainer
explainer = shap.Explainer(pred_pipeline)

# 取少量樣本 (例如 5 筆) 進行解釋，避免記憶體不足或跑太久
sample_texts = [t[:2000] for t in X_test[:5]]
shap_values = explainer(sample_texts)

# 繪圖 (在 Jupyter Notebook 中可直接顯示，若為腳本則需儲存)
shap.plots.text(shap_values)
print("SHAP values generated successfully.")

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

def plot_save_roc_curve(y_true, y_probs, save_path="roc_curve.png"):
    """
    繪製並儲存 ROC Curve
    """
    # 計算 FPR, TPR 與閾值
    fpr, tpr, thresholds = roc_curve(y_true, y_probs)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 6))
    
    # 繪製 ROC 曲線
    plt.plot(fpr, tpr, color='darkorange', lw=2, 
             label=f'ROC curve (AUROC = {roc_auc:.4f})')
    
    # 繪製對角線 (隨機猜測線)
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    
    # 圖表設定
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
    plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
    plt.title('Receiver Operating Characteristic (ROC) - Final Test Set', fontsize=14)
    plt.legend(loc="lower right", fontsize=12)
    plt.grid(alpha=0.3)
    
    # 儲存與顯示
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"\nROC Curve saved to: {save_path}")
    plt.show()

# ==========================================
# 在程式碼最後執行繪圖
# ==========================================
print("\nGenerating ROC Curve...")
# y_test: 測試集的真實標籤
# test_probs: 測試集的預測機率 (Class 1)
plot_save_roc_curve(y_test, test_probs, save_path="bluebert_roc_curve.png")  # 檔名請替換為您的模型名稱